### Define X and y

In [95]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier


df = pd.read_csv("../data/raw/student_mental_health.csv")


# remove Student id because as an identifier it is a useless feature
X = df.drop(columns=[
    "Student_ID",
    "Mental_Health_Risk"
])
# target
y = df["Mental_Health_Risk"]


print("X shape : ", X.shape)
print("y shape : ", y.shape)

X shape :  (2000, 15)
y shape :  (2000,)


### Split train & test

In [96]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train , y_test = train_test_split(
    X, y , test_size=0.2, stratify=y, random_state=42
)

print("Training: ", X_train.shape) #80%
print("Testing: ", X_test.shape) #20%

# the test set must remain untouchable during preprcocessing , decisions, model selection, hyperparameter tuning    


Training:  (1600, 15)
Testing:  (400, 15)


### Verify stratification

In [97]:
print("Training target distribution (%) : ")
print(y_train.value_counts(normalize=True) * 100)
# normalize transforme les comptes en proportions entre 0 et 1
print("Training target distribution (%) : ")
print(y_test.value_counts(normalize=True) * 100)

Training target distribution (%) : 
Mental_Health_Risk
High      34.0
Medium    33.0
Low       33.0
Name: proportion, dtype: float64
Training target distribution (%) : 
Mental_Health_Risk
High      34.0
Medium    33.0
Low       33.0
Name: proportion, dtype: float64


In [98]:
print("Features type")
print(X_train.dtypes)

print("\n Missing values")
print(X_train.isna().sum())

Features type
Age                                 int64
Gender                             object
Year_of_Study                       int64
Sleep_Hours                       float64
Study_Hours_Per_Week              float64
GPA                               float64
Social_Media_Hours                float64
Screen_Time_Hours                 float64
Physical_Activity                  object
Financial_Stress                  float64
Family_Support                    float64
Attendance_Rate                   float64
Part_Time_Job                      object
Diet_Quality                       object
Previous_Mental_Health_History     object
dtype: object

 Missing values
Age                                0
Gender                             0
Year_of_Study                      0
Sleep_Hours                       54
Study_Hours_Per_Week               0
GPA                               36
Social_Media_Hours                48
Screen_Time_Hours                  0
Physical_Activity            

### Traitement des valeurs manquantes

Identifier les valeurs ordinales


In [99]:
features_to_inspect = [
    "Year_of_Study",
    "Physical_Activity",
    "Financial_Stress",
    "Family_Support",
    "Diet_Quality",
    "Part_Time_Job",
    "Previous_Mental_Health_History",
    "Gender"
]

for col in features_to_inspect:
    print(f"\n{col}")
    print("Unique values:", X_train[col].dropna().unique())


Year_of_Study
Unique values: [2 1 3 4 5]

Physical_Activity
Unique values: ['Moderate' 'High' 'Low']

Financial_Stress
Unique values: [ 7.3  4.4  5.4  7.5  7.2  6.3  9.4  7.6  7.4  2.6  5.3  1.8  7.8  6.
  6.9  5.8  4.7  5.6  4.9  7.7  7.   4.8  8.5  9.7  7.1  9.   1.9  8.9
  6.6  3.9  8.4  5.   9.1  4.6  6.2  4.   2.5  8.1  3.2  8.7  2.1  5.7
  3.1  3.5  3.7  4.1  2.4  6.1  6.7  9.2  4.5  9.5  9.3  3.   4.2  8.8
  7.9  3.4  4.3  3.3  2.7  8.   2.9  2.8  2.   5.2  6.4  6.8  8.2  5.9
  8.6  1.5  5.5  2.3  3.8  8.3  6.5  9.6  5.1  2.2  9.9  1.4  3.6  1.6
  9.8  1.7 10. ]

Family_Support
Unique values: [7.7 7.2 7.9 4.8 7.8 8.8 4.9 6.  8.  8.1 5.9 4.7 6.5 4.  5.3 5.8 5.4 6.1
 2.3 8.5 7.4 6.8 9.8 5.7 1.7 4.5 8.2 6.7 4.4 5.  3.3 3.5 8.6 9.6 5.1 8.7
 9.5 5.6 3.4 2.1 4.3 3.7 7.1 6.2 7.3 9.3 3.1 4.6 7.6 9.2 6.6 3.6 8.3 7.5
 9.4 5.5 8.9 6.9 3.8 9.  2.4 7.  3.  2.7 6.3 5.2 8.4 2.6 2.  4.2 9.9 9.7
 2.8 3.2 2.9 6.4 3.9 9.1 4.1 1.9 2.5 2.2]

Diet_Quality
Unique values: ['Average' 'Poor' 'Good']

Pa

Define imputation startegy

In [100]:
from sklearn.impute import SimpleImputer

numerical_features = [
    "Age",
    "Year_of_Study",
    "Sleep_Hours",
    "Study_Hours_Per_Week",
    "GPA",
    "Social_Media_Hours",
    "Screen_Time_Hours",
    "Financial_Stress",
    "Family_Support",
    "Attendance_Rate"
]

nominal_features = [
    "Gender",
    "Part_Time_Job",
    "Previous_Mental_Health_History"
]

ordinal_features = [
    "Physical_Activity",
    "Diet_Quality"
]
# verifier que la somme des features est égale à la somme des columns de train set
assert set(
    numerical_features + nominal_features + ordinal_features
) == set(X_train.columns)

print("Numerical: ", len(numerical_features))
print("Nominal: ", len(nominal_features))
print("Ordinal: ", len(ordinal_features))


numeric_imputer = SimpleImputer(strategy="median")
nominal_imputer = SimpleImputer(strategy="most_frequent")
ordinal_imputer = SimpleImputer(strategy="most_frequent")

numeric_imputer.fit(X_train[numerical_features]) #calcue une mediane , most frequent categories, scalling statistics, encoding categories, pour chacune des dix colonnes numeriques d'après le X train
nominal_imputer.fit(X_train[nominal_features])
ordinal_imputer.fit(X_train[ordinal_features])


print(
    numeric_imputer.statistics_
)


median_report = pd.Series(
    numeric_imputer.statistics_,
    index=numerical_features,
    name="Training median"
)

display(median_report)

Numerical:  10
Nominal:  3
Ordinal:  2
[21.    2.    6.7  19.8   2.56  6.    8.5   6.1   6.7  85.1 ]


Age                     21.00
Year_of_Study            2.00
Sleep_Hours              6.70
Study_Hours_Per_Week    19.80
GPA                      2.56
Social_Media_Hours       6.00
Screen_Time_Hours        8.50
Financial_Stress         6.10
Family_Support           6.70
Attendance_Rate         85.10
Name: Training median, dtype: float64

Appliquer l'imputation sur les valeurs d'entrainement

In [101]:
X_train_numeric_imputed = numeric_imputer.transform(
    X_train[numerical_features]
)

X_train_numeric_imputed = pd.DataFrame(
    X_train_numeric_imputed,
    columns=numerical_features,
    index=X_train.index
)

X_train_numeric_imputed.isna().sum()




print("Nominal imputation values:")
print(dict(zip(nominal_features, nominal_imputer.statistics_)))

print("Ordinal imputaion values:")
print(dict(zip(ordinal_features, ordinal_imputer.statistics_)))



Nominal imputation values:
{'Gender': 'Female', 'Part_Time_Job': 'No', 'Previous_Mental_Health_History': 'No'}
Ordinal imputaion values:
{'Physical_Activity': 'Moderate', 'Diet_Quality': 'Average'}


### Build the preprocessing Pipeline

In [102]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

nominal_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

ordinal_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(
        categories=[ # the order of categories must match the order of ordinal features
            ["Low","Moderate","High"],
            ["Poor", "Average", "Good"]
        ],
        handle_unknown="use_encoded_value", # used so when the encoder doesnt encounter a new unknown caterory it will not raise an exception error 
        unknown_value=-1
    ))]
)


# combine everything in columntransformer 
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numerical_features),
        ("nom", nominal_pipeline, nominal_features),
        ("ord", ordinal_pipeline, ordinal_features)
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
# y_train_processed = preprocessor.fit_transform(y_train)

print("Original shape:", X_train.shape)
print("Processed Shape: ", X_train_processed.shape)
print(
    "Missing values after preprocessing:",
    np.isnan(X_train_processed).sum()
)

# display(pd.DataFrame(X_train_processed))

# features_names = preprocessor.get_feature_names_out()

# for f in features_names:
#     print(f)




Original shape: (1600, 15)
Processed Shape:  (1600, 19)
Missing values after preprocessing: 0


### train importation

In [103]:
from pathlib import Path
import joblib

# Assuming notebook is executed from the project root
PROJECT_ROOT = Path.cwd().parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

train_df = X_train.copy()
train_df["Mental_Health_Risk"] = y_train

test_df = X_test.copy()
test_df["Mental_Health_Risk"] = y_test

train_df.to_csv(PROCESSED_DIR / "train.csv", index=False)
test_df.to_csv(PROCESSED_DIR / "test.csv", index=False)

joblib.dump(
    preprocessor,
    MODELS_DIR / "preprocessor.joblib"
)

print("All artifacts saved successfully!")



df_test = pd.read_csv(PROCESSED_DIR / "train.csv")
# df_test.isna().sum()
df_test.isnull().sum()
# df_test.isna().sum()

All artifacts saved successfully!


Age                                0
Gender                             0
Year_of_Study                      0
Sleep_Hours                       54
Study_Hours_Per_Week               0
GPA                               36
Social_Media_Hours                48
Screen_Time_Hours                  0
Physical_Activity                  0
Financial_Stress                  76
Family_Support                    36
Attendance_Rate                   39
Part_Time_Job                      0
Diet_Quality                      55
Previous_Mental_Health_History     0
Mental_Health_Risk                 0
dtype: int64